# 01 — Synthetic Telemetry and Command Data Generation


Data Sources

* Telemetry dataset: Time-stamped environmental readings, device health indicators, actuator states, and water-management measurements.
* Command dataset: Time-stamped operator actions, including pump and fan activation or deactivation, device inspections, and situations where no intervention is necessary.

Imports and Configuration

* We will use NumPy for synthetic numerical Data Generation, pandas for structured time-series data, and Matplotlib and Seaborn for exploratory visualisations.

## Import libraries

In [1]:
# Numerical Operations and random data Generation
import numpy as np
import pandas as pd

# Visualisation libraries for inspecting generated Datasets
import matplotlib.pyplot as plt
import seaborn as sns

# File system utilities for saving generated Datasets
from pathlib import Path

# Display complete column lists and rRadable DataFrame previews
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Apply a consistent visual theme to future Charts
sns.set_theme(style="whitegrid")

print("Imports Completed Successfully")

Matplotlib is building the font cache; this may take a moment.


Imports Completed Successfully


## Project Configuration

In [2]:
# Reproducibility settings and the Simulated operating Period
RANDOM_SEED = 42
SIMULATION_DAYS = 30
READING_INTERVAL_MINUTES = 15

# Number of simulated Devices in the Hydroponic Facility
NUMBER_OF_DEVICES = 12

# Random number generator used throughout this Notebook
rng = np.random.default_rng(RANDOM_SEED)

# Project root relative to this notebooks location
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Directory for Generated Datasets
DATA_DIRECTORY = PROJECT_ROOT / "data" / "generated"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Calculate the number of Telemetry Observations per Device
READINGS_PER_DEVICE = (
    SIMULATION_DAYS * 24 * 60 // READING_INTERVAL_MINUTES
)

TOTAL_TELEMETRY_READINGS = NUMBER_OF_DEVICES * READINGS_PER_DEVICE

print(f"Project Root: {PROJECT_ROOT}")
print(f"Random Seed: {RANDOM_SEED}")
print(f"Simulation Period: {SIMULATION_DAYS} days")
print(f"Devices: {NUMBER_OF_DEVICES}")
print(f"Readings per Device: {READINGS_PER_DEVICE}")
print(f"Expected Telemetry Readings: {TOTAL_TELEMETRY_READINGS:,}")
print(f"Output Directory: {DATA_DIRECTORY}")

Project Root: /Users/quendrew/Desktop/Smart-X-ML
Random Seed: 42
Simulation Period: 30 days
Devices: 12
Readings per Device: 2880
Expected Telemetry Readings: 34,560
Output Directory: /Users/quendrew/Desktop/Smart-X-ML/data/generated


## Create Device Metadata

In [3]:
# Define the Device Categories represented in the Simulated Facility
device_definitions = [
    ("SX-ENV-001", "NF-001", "ZONE-A", "Environmental"),
    ("SX-ENV-002", "NF-002", "ZONE-A", "Environmental"),
    ("SX-ENV-003", "NF-003", "ZONE-A", "Environmental"),
    ("SX-WTR-001", "NF-004", "ZONE-A", "Water Management"),
    ("SX-WTR-002", "NF-005", "ZONE-A", "Water Management"),
    ("SX-ENV-004", "NF-006", "ZONE-B", "Environmental"),
    ("SX-ENV-005", "NF-007", "ZONE-B", "Environmental"),
    ("SX-ENV-006", "NF-008", "ZONE-B", "Environmental"),
    ("SX-WTR-003", "NF-009", "ZONE-B", "Water Management"),
    ("SX-WTR-004", "NF-010", "ZONE-B", "Water Management"),
    ("SX-ENV-007", "NF-011", "ZONE-C", "Environmental"),
    ("SX-WTR-005", "NF-012", "ZONE-C", "Water Management"),
]

devices_df = pd.DataFrame(
    device_definitions,
    columns=["device_id", "node_id", "zone", "device_category"]
)

# Device Health baselines for the Simulation
devices_df["baseline_health"] = rng.uniform(
    0.88, 0.99, size=len(devices_df)
).round(3)

devices_df["device_status"] = "ONLINE"

display(devices_df)

,device_id,node_id,zone,device_category,baseline_health,device_status
0,SX-ENV-001,NF-001,ZONE-A,Environmental,0.965,ONLINE
1,SX-ENV-002,NF-002,ZONE-A,Environmental,0.928,ONLINE
2,SX-ENV-003,NF-003,ZONE-A,Environmental,0.974,ONLINE
3,SX-WTR-001,NF-004,ZONE-A,Water Management,0.957,ONLINE
4,SX-WTR-002,NF-005,ZONE-A,Water Management,0.890,ONLINE
5,SX-ENV-004,NF-006,ZONE-B,Environmental,0.987,ONLINE
6,SX-ENV-005,NF-007,ZONE-B,Environmental,0.964,ONLINE
7,SX-ENV-006,NF-008,ZONE-B,Environmental,0.966,ONLINE
8,SX-WTR-003,NF-009,ZONE-B,Water Management,0.894,ONLINE
9,SX-WTR-004,NF-010,ZONE-B,Water Management,0.930,ONLINE


## Telemtry Generation

In [4]:
# Establish a Common UTC Timeline
simulation_start = pd.Timestamp("2026-01-01T00:00:00Z")

timestamps = pd.date_range(
    start=simulation_start,
    periods=READINGS_PER_DEVICE,
    freq=f"{READING_INTERVAL_MINUTES}min"
)

telemetry_records = []

for device in devices_df.itertuples(index=False):
    phase_offset = rng.uniform(0, 2 * np.pi)
    temperature_baseline = rng.normal(24.0, 1.5)
    humidity_baseline = rng.normal(65.0, 5.0)
    moisture_baseline = rng.normal(58.0, 7.0)
    ph_baseline = rng.normal(6.0, 0.2)
    ec_baseline = rng.normal(1.8, 0.2)

    # Create a Gradual environmental Drift across the Simulation Period
    drift = np.linspace(
        0,
        rng.normal(0, 1.5),
        READINGS_PER_DEVICE
    )

    # Model a daily Temperature cycle plus Gradual Drift and random Noise
    hours = np.arange(READINGS_PER_DEVICE) * (
        READING_INTERVAL_MINUTES / 60
    )
    daily_cycle = 3.0 * np.sin(
        (2 * np.pi * hours / 24) + phase_offset
    )

    temperature = (
        temperature_baseline
        + daily_cycle
        + drift
        + rng.normal(0, 0.8, READINGS_PER_DEVICE)
    )

    humidity = (
        humidity_baseline
        - 0.7 * daily_cycle
        + rng.normal(0, 3.0, READINGS_PER_DEVICE)
    )

    moisture = (
        moisture_baseline
        + rng.normal(0, 4.0, READINGS_PER_DEVICE)
        - np.linspace(0, rng.uniform(0, 12), READINGS_PER_DEVICE)
    )

    # Occasionally simulate abnormal conditions lasting several readings.
    if rng.random() < 0.65:
        event_start = int(rng.integers(0, READINGS_PER_DEVICE - 24))
        event_duration = int(rng.integers(4, 25))
        event_end = min(event_start + event_duration, READINGS_PER_DEVICE)

        event_type = rng.choice(
            ["high_temperature", "low_moisture", "high_humidity"]
        )

        if event_type == "high_temperature":
            temperature[event_start:event_end] += rng.uniform(3.0, 6.0)
        elif event_type == "low_moisture":
            moisture[event_start:event_end] -= rng.uniform(12.0, 22.0)
        else:
            humidity[event_start:event_end] += rng.uniform(12.0, 20.0)

    # Keep generated Measurements within reasonable sensor Ranges
    temperature = np.clip(temperature, 8, 45)
    humidity = np.clip(humidity, 15, 100)
    moisture = np.clip(moisture, 0, 100)

    is_water_device = device.device_category == "Water Management"

    if is_water_device:
        ph_values = np.clip(
            ph_baseline + rng.normal(0, 0.12, READINGS_PER_DEVICE),
            3.5, 9.5
        )
        ec_values = np.clip(
            ec_baseline + rng.normal(0, 0.15, READINGS_PER_DEVICE),
            0.1, 4.5
        )
        flow_values = np.clip(
            rng.normal(1.2, 0.25, READINGS_PER_DEVICE),
            0, 3.0
        )
    else:
        # Missing values indicate Measurements not provided by this Device
        ph_values = np.full(READINGS_PER_DEVICE, np.nan)
        ec_values = np.full(READINGS_PER_DEVICE, np.nan)
        flow_values = np.full(READINGS_PER_DEVICE, np.nan)

    # Simulate changing Health and Battery Levels
    health_values = np.clip(
        device.baseline_health
        + rng.normal(0, 0.015, READINGS_PER_DEVICE),
        0.5, 1.0
    )
    battery_values = np.clip(
        100 - np.linspace(0, rng.uniform(0, 18), READINGS_PER_DEVICE)
        + rng.normal(0, 1.5, READINGS_PER_DEVICE),
        0, 100
    )

    # Store one Telemetry record per device and Timestamp
    for i, timestamp in enumerate(timestamps):
        telemetry_records.append({
            "timestamp": timestamp,
            "device_id": device.device_id,
            "node_id": device.node_id,
            "zone": device.zone,
            "device_category": device.device_category,
            "temperature_c": round(float(temperature[i]), 2),
            "humidity_pct": round(float(humidity[i]), 2),
            "moisture_pct": round(float(moisture[i]), 2),
            "water_ph": (
                round(float(ph_values[i]), 2)
                if not np.isnan(ph_values[i]) else np.nan
            ),
            "water_ec_ms_cm": (
                round(float(ec_values[i]), 2)
                if not np.isnan(ec_values[i]) else np.nan
            ),
            "water_flow_l_min": (
                round(float(flow_values[i]), 2)
                if not np.isnan(flow_values[i]) else np.nan
            ),
            "device_health": round(float(health_values[i]), 3),
            "battery_pct": round(float(battery_values[i]), 2),
        })

telemetry_df = pd.DataFrame(telemetry_records)

# Records are Sorted by Time and Device
telemetry_df = telemetry_df.sort_values(
    ["timestamp", "device_id"]
).reset_index(drop=True)

print(f"Generated Telemetry Records: {len(telemetry_df):,}")
print(f"Unique Devices: {telemetry_df['device_id'].nunique()}")
print(
    f"Time Range: {telemetry_df['timestamp'].min()} "
    f"to {telemetry_df['timestamp'].max()}"
)

display(telemetry_df.head(10))

Generated Telemetry Records: 34,560
Unique Devices: 12
Time Range: 2026-01-01 00:00:00+00:00 to 2026-01-30 23:45:00+00:00


,timestamp,device_id,node_id,zone,device_category,temperature_c,humidity_pct,moisture_pct,water_ph,water_ec_ms_cm,water_flow_l_min,device_health,battery_pct
0,2026-01-01 00:00:00+00:00,SX-ENV-001,NF-001,ZONE-A,Environmental,23.29,71.41,49.59,NaN,NaN,NaN,0.974,100.00
1,2026-01-01 00:00:00+00:00,SX-ENV-002,NF-002,ZONE-A,Environmental,20.92,63.62,53.99,NaN,NaN,NaN,0.932,100.00
2,2026-01-01 00:00:00+00:00,SX-ENV-003,NF-003,ZONE-A,Environmental,27.71,66.85,58.77,NaN,NaN,NaN,0.957,100.00
3,2026-01-01 00:00:00+00:00,SX-ENV-004,NF-006,ZONE-B,Environmental,25.70,73.08,56.39,NaN,NaN,NaN,0.993,100.00
4,2026-01-01 00:00:00+00:00,SX-ENV-005,NF-007,ZONE-B,Environmental,24.84,58.19,54.72,NaN,NaN,NaN,0.952,100.00
5,2026-01-01 00:00:00+00:00,SX-ENV-006,NF-008,ZONE-B,Environmental,27.08,57.70,39.63,NaN,NaN,NaN,0.960,100.00
6,2026-01-01 00:00:00+00:00,SX-ENV-007,NF-011,ZONE-C,Environmental,22.38,66.55,60.94,NaN,NaN,NaN,0.915,98.00
7,2026-01-01 00:00:00+00:00,SX-WTR-001,NF-004,ZONE-A,Water Management,27.59,65.17,57.15,6.24,1.38,1.52,0.951,99.32
8,2026-01-01 00:00:00+00:00,SX-WTR-002,NF-005,ZONE-A,Water Management,22.89,61.70,61.55,6.04,1.75,0.86,0.900,99.99
9,2026-01-01 00:00:00+00:00,SX-WTR-003,NF-009,ZONE-B,Water Management,25.68,52.31,51.34,5.89,1.75,1.17,0.879,100.00


## Validate Telemetry

In [5]:
# Check the structural Properties before saving the Dataset
assert telemetry_df["device_id"].nunique() == NUMBER_OF_DEVICES
assert len(telemetry_df) == TOTAL_TELEMETRY_READINGS
assert telemetry_df["timestamp"].is_monotonic_increasing
assert not telemetry_df.duplicated(["timestamp", "device_id"]).any()

# Verify the Environmental Measurements remain within expected Bounds
assert telemetry_df["temperature_c"].between(8, 45).all()
assert telemetry_df["humidity_pct"].between(15, 100).all()
assert telemetry_df["moisture_pct"].between(0, 100).all()
assert telemetry_df["device_health"].between(0.5, 1.0).all()
assert telemetry_df["battery_pct"].between(0, 100).all()

print("Telemetry Validation Passed")
print("\nMissing values by Column:")
display(telemetry_df.isna().sum().to_frame("missing_count"))

Telemetry Validation Passed

Missing values by Column:


,missing_count
timestamp,0
device_id,0
node_id,0
zone,0
device_category,0
temperature_c,0
humidity_pct,0
moisture_pct,0
water_ph,20160
water_ec_ms_cm,20160


## Save Telemetry and Device Registry

In [6]:
# Save Generated Datasets
telemetry_path = DATA_DIRECTORY / "synthetic_telemetry.csv"
devices_path = DATA_DIRECTORY / "device_registry.csv"

telemetry_df.to_csv(telemetry_path, index=False)
devices_df.to_csv(devices_path, index=False)

print(f"Telemetry Saved to: {telemetry_path}")
print(f"Device Registry Saved to: {devices_path}")

Telemetry Saved to: /Users/quendrew/Desktop/Smart-X-ML/data/generated/synthetic_telemetry.csv
Device Registry Saved to: /Users/quendrew/Desktop/Smart-X-ML/data/generated/device_registry.csv
